In [1]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import faiss, numpy as np, json, torch

# Forcer CPU et limiter les threads pour économiser la RAM
torch.set_num_threads(2)

# Embedding model — léger (80MB)
print("Chargement embedding model...")
embedding_model = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1",
    device="cpu"
)
print("OK")

print("Chargement RoBERTa...")
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_name,
    torch_dtype=torch.float32
    # low_cpu_mem_usage retiré — nécessite accelerate
)
qa_model.eval()
print("OK")

# KB + Index FAISS
print("Chargement KB et index...")
with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

index = faiss.read_index("ensa_faiss_v2.index")
print("OK")

print(f"\nKB         : {len(ensa_data)} entrées")
print(f"Index FAISS : {index.ntotal} vecteurs")
print(f"Alignement  : {len(ensa_data) == index.ntotal}")

# Vérifier la RAM disponible
import psutil
ram = psutil.virtual_memory()
print(f"RAM utilisée : {ram.percent}% ({ram.used // 1024**2} MB / {ram.total // 1024**2} MB)")

Chargement embedding model...


C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OK
Chargement RoBERTa...


C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OK
Chargement KB et index...
OK

KB         : 320 entrées
Index FAISS : 42 vecteurs
Alignement  : False
RAM utilisée : 93.1% (7020 MB / 7541 MB)


In [9]:
# Reconstruire l'index FAISS avec passages normalisés
import unicodedata

def normaliser(texte):
    return unicodedata.normalize('NFD', texte)\
                      .encode('ascii', 'ignore')\
                      .decode('utf-8')

passages_alignes = [
    normaliser(item["question"] + " " + item["answer"])
    for item in ensa_data
]

print("Encodage avec normalisation...")
embeddings = embedding_model.encode(
    passages_alignes,
    show_progress_bar=True,
    batch_size=32
)
embeddings = embeddings.astype(np.float32)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, "ensa_faiss_v3.index")
print(f"Index v3 prêt : {index.ntotal} vecteurs")

Encodage avec normalisation...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Index v3 prêt : 320 vecteurs


In [15]:
import unicodedata, torch

SEUIL_COSINUS = 0.40

# ── Normalisation ────────────────────────────────────────────
def normaliser(texte):
    return unicodedata.normalize('NFD', texte)\
                      .encode('ascii', 'ignore')\
                      .decode('utf-8')

# ── Retriever ────────────────────────────────────────────────
def retrieve_v2(query, top_k=3):
    query_normalisee = normaliser(query)
    query_vec = embedding_model.encode([query_normalisee]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(ensa_data):
            results.append({
                "rank": i + 1,
                "score_cosinus": float(scores[0][i]),
                "answer": ensa_data[idx]["answer"],
                "question": ensa_data[idx]["question"],
                "category": ensa_data[idx]["category"]
            })
    return results

# ── Reader RoBERTa ───────────────────────────────────────────
def read_answer(question, context):
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )
    score = float(torch.softmax(outputs.start_logits, dim=1).max())
    return answer, score

# ── Score confiance ──────────────────────────────────────────
def calculer_confiance(score_cosinus, score_qa):
    if score_cosinus < SEUIL_COSINUS:
        return 0
    return min(round((0.6 * score_cosinus + 0.4 * score_qa) * 100), 99)

# ── Pipeline complet ─────────────────────────────────────────
def chatbot_ensa(query, top_k=3):
    passages = retrieve_v2(query, top_k)
    if not passages:
        return {"reponse": "Internal error.", "confiance": 0}
    meilleur = passages[0]
    if meilleur["score_cosinus"] < SEUIL_COSINUS:
        return {
            "reponse": "Information not available. Please contact contact@ensa-fes.ma",
            "confiance": 0,
            "category": "hors_kb",
            "source": None
        }
    contexte = meilleur["answer"]
    reponse, score_qa = read_answer(query, contexte)
    confiance = calculer_confiance(meilleur["score_cosinus"], score_qa)
    return {
        "reponse": reponse,
        "source": meilleur["question"],
        "confiance": confiance,
        "category": meilleur["category"],
        "score_cosinus": round(meilleur["score_cosinus"], 4),
        "score_qa": round(score_qa, 4),
        "passage": meilleur["answer"]
    }

print("Pipeline v3 chargé avec normalisation !")

Pipeline v3 chargé avec normalisation !


In [16]:
import pandas as pd

# Questions qui DOIVENT avoir une réponse — couvrent toutes les catégories
questions_in_kb = [
    # Admissions
    ("How to get admitted to ENSA Fes?",             "admissions"),
    ("What baccalaureate is required for ENSA Fes?", "admissions"),
    # Filières
    ("What engineering programs are available?",      "filieres"),
    ("What is the Computer Science program about?",   "filieres"),
    # Scolarité
    ("How long is the engineering program?",          "scolarite"),
    ("When do exams take place at ENSA Fes?",         "scolarite"),
    # Stages
    ("Are internships mandatory at ENSA Fes?",        "stages"),
    ("What is the final year project PFE?",           "stages"),
    # Clubs
    ("What clubs are available at ENSA Fes?",         "clubs"),
    ("How to join a club at ENSA Fes?",               "clubs"),
    # Contacts
    ("Where is ENSA Fes located?",                    "contacts"),
    ("What is the contact email of ENSA Fes?",        "contacts"),
    # Départements
    ("Who are the professors in computer science?",   "departements"),
    # Services
    ("How to access student email at ENSA Fes?",      "services"),
    # Bourses
    ("Are there scholarships available at ENSA Fes?", "bourses"),
]

resultats_in = []
print("Test questions dans KB...\n")

for question, cat_attendue in questions_in_kb:
    res = chatbot_ensa(question)
    c = res["confiance"]
    badge = "VERT  " if c > 70 else "ORANGE" if c > 40 else "ROUGE "
    correct = res.get("category") == cat_attendue

    resultats_in.append({
        "Question": question[:40] + "...",
        "Confiance": f"{c}%",
        "Badge": badge,
        "Cos.": res.get("score_cosinus", "N/A"),
        "QA": res.get("score_qa", "N/A"),
        "Catégorie": res.get("category", "N/A"),
        "Attendue": cat_attendue,
        "Correct": "OK" if correct else "ERR"
    })

df_in = pd.DataFrame(resultats_in)
print(df_in[["Question","Confiance","Badge","Cos.","QA","Catégorie","Correct"]].to_string(index=False))

verts  = len([r for r in resultats_in if r["Badge"] == "VERT  "])
oranges = len([r for r in resultats_in if r["Badge"] == "ORANGE"])
rouges = len([r for r in resultats_in if r["Badge"] == "ROUGE "])
corrects = len([r for r in resultats_in if r["Correct"] == "OK"])

print(f"\nRésultats KB ({len(questions_in_kb)} questions) :")
print(f"  VERT   (>70%) : {verts}")
print(f"  ORANGE (40-70%) : {oranges}")
print(f"  ROUGE  (<40%) : {rouges}")
print(f"  Bonne catégorie : {corrects}/{len(questions_in_kb)}")

Test questions dans KB...

                                   Question Confiance  Badge   Cos.     QA    Catégorie Correct
        How to get admitted to ENSA Fes?...       71% VERT   0.7609 0.6433   admissions      OK
What baccalaureate is required for ENSA ...       65% ORANGE 0.7944 0.4333   admissions      OK
What engineering programs are available?...       57% ORANGE 0.6028 0.5262     filieres      OK
What is the Computer Science program abo...       69% ORANGE 0.4760 0.9993     filieres      OK
    How long is the engineering program?...       75% VERT   0.6412 0.9201     filieres     ERR
   When do exams take place at ENSA Fes?...       70% ORANGE 0.7710 0.5955    scolarite      OK
  Are internships mandatory at ENSA Fes?...       77% VERT   0.9214 0.5505       stages      OK
     What is the final year project PFE?...       53% ORANGE 0.7395 0.2157       stages      OK
   What clubs are available at ENSA Fes?...       83% VERT   0.7594 0.9449        clubs      OK
         How 

In [17]:
# Questions qui NE doivent PAS avoir de réponse dans la KB
questions_hors_kb = [
    "What is the best restaurant near ENSA Fes?",
    "How to learn machine learning from scratch?",
    "What is the weather in Fes today?",
    "Who won the World Cup in 2022?",
    "What is the price of Bitcoin today?",
    "How to cook a tajine?",
]

resultats_hors = []
print("Test questions hors KB...\n")

for question in questions_hors_kb:
    res = chatbot_ensa(question)
    c = res["confiance"]
    detecte = res.get("category") == "hors_kb"

    resultats_hors.append({
        "Question": question[:45] + "...",
        "Confiance": f"{c}%",
        "Détecté hors KB": "OK" if detecte else "ERREUR",
        "Cos.": res.get("score_cosinus", "N/A"),
    })

df_hors = pd.DataFrame(resultats_hors)
print(df_hors.to_string(index=False))

bien_detectes = len([r for r in resultats_hors if r["Détecté hors KB"] == "OK"])
print(f"\nQuestions hors KB bien détectées : {bien_detectes}/{len(questions_hors_kb)}")

Test questions hors KB...

                                      Question Confiance Détecté hors KB    Cos.
 What is the best restaurant near ENSA Fes?...       61%          ERREUR  0.7154
How to learn machine learning from scratch?...        0%              OK     N/A
          What is the weather in Fes today?...       62%          ERREUR  0.4087
             Who won the World Cup in 2022?...        0%              OK     N/A
        What is the price of Bitcoin today?...        0%              OK     N/A
                      How to cook a tajine?...        0%              OK     N/A

Questions hors KB bien détectées : 4/6


In [18]:
print("="*55)
print("VALIDATION FINALE DU PIPELINE")
print("="*55)

total_kb    = len(questions_in_kb)
total_hors  = len(questions_hors_kb)

score_kb    = verts + oranges  # questions KB avec réponse
score_hors  = bien_detectes    # questions hors KB bien rejetées

print(f"\nQuestions dans KB  : {score_kb}/{total_kb} répondues correctement")
print(f"Questions hors KB  : {score_hors}/{total_hors} bien détectées")
print(f"Catégories correctes : {corrects}/{total_kb}")

# Verdict
if score_kb >= 12 and score_hors >= 5:
    print("\nVERDICT : Pipeline validé — prêt pour le frontend Gradio !")
elif score_kb >= 10 and score_hors >= 4:
    print("\nVERDICT : Pipeline acceptable — on peut passer au frontend.")
else:
    print("\nVERDICT : Amélioration nécessaire — enrichir la KB ou ajuster le seuil.")

print("="*55)

VALIDATION FINALE DU PIPELINE

Questions dans KB  : 15/15 répondues correctement
Questions hors KB  : 4/6 bien détectées
Catégories correctes : 12/15

VERDICT : Pipeline acceptable — on peut passer au frontend.


In [7]:
# Voir exactement ce que le retriever trouve pour les questions ROUGE
questions_rouge = [
    "What engineering programs are available?",
    "How long is the engineering program?",
    "Are internships mandatory at ENSA Fes?",
    "What is the final year project PFE?",
    "Who are the professors in computer science?",
]

print("DIAGNOSTIC — top 3 passages pour chaque question ROUGE\n")
for q in questions_rouge:
    passages = retrieve_v2(q, top_k=3)
    print(f"Q: {q}")
    for p in passages:
        print(f"  Rank {p['rank']} | cos={p['score_cosinus']:.4f} | cat={p['category']:<15} | {p['question'][:55]}")
    print()

DIAGNOSTIC — top 3 passages pour chaque question ROUGE

Q: What engineering programs are available?
  Rank 1 | cos=0.2591 | cat=admissions      | What is the Integrated Preparatory Cycle (CPI) at ENSA 
  Rank 2 | cos=0.2545 | cat=admissions      | How can I prepare for the CNC exam to maximize my chanc
  Rank 3 | cos=0.2525 | cat=admissions      | What is the duration of the engineering program at ENSA

Q: How long is the engineering program?
  Rank 1 | cos=0.2131 | cat=admissions      | What is the role of the admissions office at ENSA Fès?
  Rank 2 | cos=0.2055 | cat=admissions      | What is the duration of the engineering program at ENSA
  Rank 3 | cos=0.1887 | cat=admissions      | Is ENSA Fès a public or private institution?

Q: Are internships mandatory at ENSA Fes?
  Rank 1 | cos=0.4372 | cat=admissions      | What is the academic calendar for the first year of CPI
  Rank 2 | cos=0.4032 | cat=admissions      | Does ENSA Fès accept BTS holders for parallel admission
  Rank 3 | c

In [8]:
from collections import Counter

with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

categories = Counter(item["category"] for item in ensa_data)

print(f"Total entrées : {len(ensa_data)}\n")
print("Répartition par catégorie :")
print("-" * 40)
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    barre = "█" * (count // 2)
    print(f"  {cat:<20} {count:>4} entrées  {barre}")

print("-" * 40)
print(f"\nCatégories manquantes attendues :")
attendues = {"admissions","filieres","scolarite","stages","clubs",
             "contacts","departements","services","bourses","evenements"}
manquantes = attendues - set(categories.keys())
for m in manquantes:
    print(f"  MANQUANTE : {m}")

Total entrées : 320

Répartition par catégorie :
----------------------------------------
  filieres               50 entrées  █████████████████████████
  departements           50 entrées  █████████████████████████
  admissions             40 entrées  ████████████████████
  scolarite              30 entrées  ███████████████
  stages                 30 entrées  ███████████████
  clubs                  30 entrées  ███████████████
  contacts               30 entrées  ███████████████
  evenements             30 entrées  ███████████████
  services               30 entrées  ███████████████
----------------------------------------

Catégories manquantes attendues :
  MANQUANTE : bourses


In [14]:
q = "Who are the professors in computer science?"
passages = retrieve_v2(q, top_k=3)
print(f"Query normalisée : {normaliser(q)}\n")
for p in passages:
    print(f"Rank {p['rank']} | cos={p['score_cosinus']:.4f} | cat={p['category']}")
    print(f"  Q: {p['question']}")
    print()

Query normalisée : Who are the professors in computer science?

Rank 1 | cos=0.4237 | cat=departements
  Q: Who are the faculty members of the Applied Mathematics department at ENSA Fès?

Rank 2 | cos=0.3997 | cat=departements
  Q: Who are the faculty members of the GEI department at ENSA Fès?

Rank 3 | cos=0.3953 | cat=departements
  Q: Who are the faculty members of the GSI department at ENSA Fès?

